# UC fibroblasts: TNC and its predicted targets (Fig 3G)

Dot plot of TNC and its NicheNet-predicted downstream targets across fibroblast subtypes of the UC atlas, with the NicheNet
regulatory potential of each target as a bar. The regulatory potentials are the TNC column of the NicheNet TF-target matrix
used in `14_nichenet_macrophage_to_fibroblast.R` and are typed into the plotting cell.

Pericytes, cycling and SELENOP+ fibroblasts are left out.

**Input**: `uc_atlas_annotated.h5ad` (06_atlas_assembly.R).

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────────
DATA_DIR   = "/path/to/uc/scrna/output"
OUTPUT_DIR = f"{DATA_DIR}/fibroblast"

In [ ]:
# Load full atlas
scrna = sc.read_h5ad(f"{DATA_DIR}/uc_atlas_annotated.h5ad")

## 2. Subset fibroblasts

In [ ]:
fib = scrna[scrna.obs["cell_category"] == "Connective"].copy()

In [ ]:
# Remove pericytes – focus on stromal fibroblast compartment
exclude_fib = ["CD36+ Peri", "RERGL+ Contr Peri"]
fib = fib[~fib.obs["cell_type_short"].isin(exclude_fib)].copy()

In [ ]:
# Ordered fibroblast subtypes (ADAMDEC1+ Fib as anchor)
ct_order = [
    "ADAMDEC1+ Fib", "OGN+RSPO3+ Fib", "T-interact Fib",
    "Activ Fib", "CXCL5+ Activ Fib",
    "NRG1+ Crypt Top Fib", "VSTM2A+ Crypt Top Fib",
    "HHIP+ Myofib", "GREM2+ Myofib"
]

# Subset to non-cycling, non-SELENOP+ fibroblasts for clean display
fib_cl = fib[~fib.obs["cell_type_short"].isin(["SELENOP+ Fib", "Cycl Fib"])].copy()
fib_cl.obs["cell_type_short"] = pd.Categorical(
    fib_cl.obs["cell_type_short"], categories=ct_order
)

## TNC downstream targets with NicheNet regulatory potential (Fig 3G)

TNC targets from the NicheNet TF-target network (`14_nichenet_macrophage_to_fibroblast.R`). Expression as dot plot; regulatory potential as the bar on the left.

In [ ]:
# TNC downstream targets + NicheNet regulation scores (from 14_nichenet_macrophage_to_fibroblast.R output)
tnc_reg_df = pd.DataFrame({
    "gene": ["EDNRA", "MMP13", "MMP3", "TIMP3", "ADAMTS4",
              "MBP", "KHDRBS1", "AR", "MMP2", "MMP9", "IL6", "TNF"],
    "tnc_reg": [0.014115, 0.012279, 0.012237, 0.011308, 0.011019,
                0.010472, 0.010289, 0.009902, 0.009145, 0.008327, 0.005794, 0.005686]
}).set_index("gene")

tnc_targets = ["TNC"] + tnc_reg_df.index.tolist()
genes_present = [g for g in tnc_targets if g in fib_cl.var_names]

In [ ]:
# Dotplot with axes swapped (genes on Y) + NicheNet bar on the left
dp = sc.pl.dotplot(
    fib_cl, var_names=genes_present, groupby="cell_type_short",
    standard_scale=None, vmin=0, vmax=30,
    swap_axes=True, return_fig=True, figsize=(5, 4)
)
dp.make_figure()
axes   = dp.get_axes()
main_ax = axes["mainplot_ax"]
fig     = main_ax.figure

# Gene order as displayed in the dotplot
gene_order = [t.get_text() for t in main_ax.get_yticklabels()]
vals = tnc_reg_df.reindex(gene_order)["tnc_reg"]
vals = vals.where(vals.index != "TNC")   # TNC itself has no reg score

# Add regulation bar axis on the left
bbox  = main_ax.get_position()
bar_w = 0.15
bar_ax = fig.add_axes(
    [bbox.x0 - bar_w - 0.16, bbox.y0 - 0.028, bar_w, bbox.height],
    sharey=main_ax
)

y = np.arange(len(gene_order))
bar_ax.barh(y, vals.values, height=0.4, edgecolor="black", linewidth=0.5)

bar_ax.set_xlabel("TNC\nregulation", fontsize=9)
bar_ax.set_xlim(0, np.nanmax(vals.values) * 1.1)
bar_ax.tick_params(axis="y", left=False, labelleft=False)
bar_ax.spines["top"].set_visible(False)
bar_ax.spines["right"].set_visible(False)
bar_ax.spines["left"].set_visible(False)
bar_ax.invert_xaxis()   # bars grow leftward from gene names

main_ax.set_ylim(-0.5, len(gene_order) - 0.5)
main_ax.invert_yaxis()
main_ax.tick_params(axis="y", labelsize=10)
main_ax.set_title("TNC Downstream Targets", fontsize=12, pad=12)

fig.canvas.draw()
plt.savefig(f"{OUTPUT_DIR}/tnc_targets_dotplot_nichenet.pdf", bbox_inches="tight", dpi=300)
plt.show()